In [ ]:
import xml.etree.ElementTree as ET
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import json
import re
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained('../models/biobert-base-cased-v1.1')
model = AutoModel.from_pretrained('../models/biobert-base-cased-v1.1')

# Set model to evaluation mode
model.eval()

# Move model to device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

In [ ]:
def get_embedding(text):
    # Tokenize the input text
    inputs = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        padding='max_length'
    )
    # Move inputs to device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Get the outputs from the model
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Extract the embeddings from the last hidden state of the [CLS] token
    embeddings = outputs.last_hidden_state[:, 0, :]  # Shape: [batch_size, hidden_size]
    return embeddings.cpu().numpy().flatten()

In [ ]:
def element_to_dict(el):
    
    children = list(el)
    if not children:
        return el.text
    result = {}
    for child in children:
        child_dict = element_to_dict(child)
        if child.tag in result:
            if not isinstance(result[child.tag], list):
                result[child.tag] = [result[child.tag]]
            result[child.tag].append(child_dict)
        else:
            result[child.tag] = child_dict
            
    return result

def xml_to_dict(element):
    
    return {element.tag: element_to_dict(element)}

def read_xml_file(file_path):
    
    tree = ET.parse(file_path)
    root = tree.getroot()
    return xml_to_dict(root)

In [ ]:
def extract_intervention_name(study_id):
    
    xml_path = f'../data/trials/{study_id[:7]}xxxx/{study_id}.xml'
    xml_dict = read_xml_file(xml_path)
    xml_string = json.dumps(xml_dict)
    pattern = r'"intervention_name":\s*"([^"]*)"'
    match = re.search(pattern, xml_string)
    
    if match:
        intervention_name = match.group(1).lower()
        return intervention_name

In [ ]:
file_path = '../data/IQVIA/filtered_trial_outcomes_with_labels.csv'
df = pd.read_csv(file_path)
df['intervention_name'] = df['studyid'].apply(extract_intervention_name)

In [ ]:
with open('../data/synthetic/generated_intervention_list.txt', 'r') as file:
    intervention_list = [line.strip() for line in file.readlines()]

intervention_list = list(dict.fromkeys(intervention_list))
df = df[df['intervention_name'].isin(intervention_list)]

In [ ]:
def load_examples():

    texts = []
    labels = []
        
    for study_id in df['studyid']:
        label = df.loc[df['studyid'] == study_id, 'label'].values[0]
        xml_path = f'../data/trials/{study_id[:7]}xxxx/{study_id}.xml'
        xml_dict = read_xml_file(xml_path)
        xml_string = json.dumps(xml_dict)
        xml_string = re.sub(r'"overall_status":\s*".*?",\s*', '', xml_string)
        xml_string = re.sub(r'"why_stopped":\s*".*?",\s*', '', xml_string)
        texts.append(xml_string)
        labels.append(label)

    return texts, labels

In [ ]:
real_texts = []
for study_id in df['studyid']:
    xml_path = f'../data/trials/{study_id[:7]}xxxx/{study_id}.xml'
    xml_dict = read_xml_file(xml_path)
    xml_string = json.dumps(xml_dict)
    xml_string = re.sub(r'"overall_status":\s*".*?",\s*', '', xml_string)
    xml_string = re.sub(r'"why_stopped":\s*".*?",\s*', '', xml_string)
    real_texts.append(xml_string)

In [ ]:
synthetic_texts = []

for i in range(3358):
    with open(f'../data/synthetic/retrieval_reasoning_reports/synthetic_clinical_report_{i}.txt', 'r') as file:
        text = file.read()
        synthetic_texts.append(text)

In [ ]:
# For real data
real_embeddings = np.array([get_embedding(text) for text in real_texts])

# For synthetic data
synthetic_embeddings = np.array([get_embedding(text) for text in synthetic_texts])

In [ ]:
# Set seed for reproducibility
np.random.seed(42)

# Function to sample pairs
def sample_pairs(embeddings, num_pairs):
    num_embeddings = embeddings.shape[0]
    indices = np.random.choice(num_embeddings, size=(num_pairs, 2), replace=True)
    pairs = embeddings[indices]
    return pairs

# Function to compute similarities
def compute_similarities(pairs):
    emb1 = pairs[:, 0, :]
    emb2 = pairs[:, 1, :]
    similarities = cosine_similarity(emb1, emb2).diagonal()
    return similarities

In [ ]:
# Sample pairs and compute similarities
real_pairs = sample_pairs(real_embeddings, 10000)
real_similarities = compute_similarities(real_pairs)

synthetic_pairs = sample_pairs(synthetic_embeddings, 10000)
synthetic_similarities = compute_similarities(synthetic_pairs)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assuming 'real_similarities' and 'synthetic_similarities' data are available
# Plot histograms with a more concentrated view
plt.figure(figsize=(12, 6))
bins = np.linspace(0.83, 1.0, 50)  # Adjust bin range to focus on the part where data is concentrated

plt.hist(real_similarities, bins=bins, alpha=0.5, label='Real Data Similarities', color='#1f77b4')  # Blue
plt.hist(synthetic_similarities, bins=bins, alpha=0.5, label='Synthetic Data Similarities', color='#ff7f0e')  # Orange

# Increase font sizes for title, labels, and legend
plt.title('Distribution of Cosine Similarities Between Pairs of Embeddings', fontsize=16)
plt.xlabel('Cosine Similarity', fontsize=14)
plt.ylabel('Frequency', fontsize=14)
plt.legend(loc='upper right', fontsize=12)
plt.grid(axis='y', alpha=0.75)

# Display the plot
plt.show()

# Calculate mean and standard deviation for real similarities
mean_real = np.mean(real_similarities)
std_real = np.std(real_similarities)
print(f"Real Data Similarities - Mean: {mean_real:.6f}, Standard Deviation: {std_real:.6f}")

# Calculate mean and standard deviation for synthetic similarities
mean_synthetic = np.mean(synthetic_similarities)
std_synthetic = np.std(synthetic_similarities)
print(f"Synthetic Data Similarities - Mean: {mean_synthetic:.6f}, Standard Deviation: {std_synthetic:.6f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

# Assuming real_embeddings and synthetic_embeddings are already defined
# real_embeddings = np.array([...])
# synthetic_embeddings = np.array([...])

# Set seed for reproducibility
np.random.seed(42)

# Modified sample_pairs function to sample from two embeddings arrays
def sample_pairs_cross(embeddings1, embeddings2, num_pairs):
    num_embeddings1 = embeddings1.shape[0]
    num_embeddings2 = embeddings2.shape[0]
    indices1 = np.random.choice(num_embeddings1, size=num_pairs, replace=True)
    indices2 = np.random.choice(num_embeddings2, size=num_pairs, replace=True)
    emb1 = embeddings1[indices1]
    emb2 = embeddings2[indices2]
    pairs = np.stack((emb1, emb2), axis=1)
    return pairs

# Define function to compute cosine similarities between pairs
def compute_similarities(pairs):
    similarities = []
    for emb1, emb2 in pairs:
        similarity = cosine_similarity(emb1.reshape(1, -1), emb2.reshape(1, -1))[0][0]
        similarities.append(similarity)
    return np.array(similarities)

# Sample 1000 pairs from real and synthetic embeddings
num_pairs = 1000
pairs_real_synthetic = sample_pairs_cross(real_embeddings, synthetic_embeddings, num_pairs)

# Compute similarities
similarities_real_synthetic = compute_similarities(pairs_real_synthetic)

In [ ]:
# Plot histogram of similarities
plt.figure(figsize=(12, 6))
bins = np.linspace(0.83, 1.0, 50)
plt.hist(similarities_real_synthetic, bins=bins, alpha=0.5, color='#1f77b4')
plt.title('Distribution of Cosine Similarities Between Real and Synthetic Embeddings', fontsize=16)
plt.xlabel('Cosine Similarity', fontsize=14)
plt.ylabel('Frequency', fontsize=14)
plt.grid(axis='y', alpha=0.75)

# Increase tick font sizes
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

plt.show()

# Calculate and print statistical measures
mean_similarity = np.mean(similarities_real_synthetic)
std_similarity = np.std(similarities_real_synthetic)

print(f"Mean Cosine Similarity: {mean_similarity:.4f}")
print(f"Standard Deviation: {std_similarity:.4f}")

In [ ]:
# Combine embeddings
all_embeddings = np.vstack((real_embeddings, synthetic_embeddings))

# Create labels: 0 for real, 1 for synthetic
labels = np.array([0]*len(real_embeddings) + [1]*len(synthetic_embeddings))

In [ ]:
# Initialize t-SNE
tsne = TSNE(n_components=2, perplexity=30, max_iter=1000, random_state=42)

# Fit and transform the embeddings
tsne_results = tsne.fit_transform(all_embeddings)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming tsne_results and labels are already defined
# tsne_results = np.array([...])
# labels = np.array([...])

# Create DataFrame for plotting
df = pd.DataFrame({
    'tsne_1': tsne_results[:, 0],
    'tsne_2': tsne_results[:, 1],
    'label': labels
})

# Map labels to text
label_mapping = {0: 'Real Data', 1: 'Synthetic Data'}
df['label_text'] = df['label'].map(label_mapping)

# Plot using seaborn
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x='tsne_1', y='tsne_2',
    hue='label_text',
    palette=sns.color_palette('bright', 2),
    data=df,
    alpha=0.7
)

# Increase font sizes for title, labels, and legend
plt.title('t-SNE Plot of Real vs Synthetic Data', fontsize=16)
plt.xlabel('t-SNE Dimension 1', fontsize=14)
plt.ylabel('t-SNE Dimension 2', fontsize=14)

# Increase legend font size
plt.legend(title='Data Type', title_fontsize=12, fontsize=12)

plt.show()
